![imagenes](logo.png)

Este código construye un **preprocesamiento unificado** donde cada grupo de variables recibe el **escalado o la codificación adecuada**; además, conserva sin tocar cualquier columna no especificada gracias a `remainder="passthrough"`. El resultado es un `DataFrame` listo para alimentar a un modelo, con nombres de columnas claros y manejo seguro de **categorías nuevas** (One-Hot: ignora; Ordinal: marca con −1).

### ¿Qué hace este código?

1) **Carga de datos**  
   Se lee el CSV `datos_escalado_y_codificacion.csv` en un DataFrame `datos`.

2) **Declaración de columnas por técnica**  
   Se agrupan las columnas según el **preprocesamiento** que recibirán:
   - `cols_std`: variables numéricas continuas para **StandardScaler** (centrar en 0 y varianza 1).
   - `cols_minmax`: variables con **rango acotado** para **MinMaxScaler** (reescala a [0,1] por defecto).
   - `cols_robust`: numéricas con **posibles outliers** para **RobustScaler** (mediana e IQR).
   - `cols_onehot`: categóricas **nominales** (sin orden) para **OneHotEncoder**.
   - `cols_ordinal`: categóricas **ordinales** (con orden) para **OrdinalEncoder** con un **orden explícito** en `categorias_ordinales`.

3) **Definición del `ColumnTransformer` (`preprocesador`)**  
   Se crea un bloque que aplica **varias transformaciones en paralelo** y concatena los resultados:
   - `("std", StandardScaler(), cols_std)`: estandariza las columnas de `cols_std`.
   - `("minmax", MinMaxScaler(), cols_minmax)`: normaliza a un rango fijo las de `cols_minmax`.
   - `("robust", RobustScaler(), cols_robust)`: escala de forma robusta las de `cols_robust`.
   - `("onehot", OneHotEncoder(...), cols_onehot)`: hace **One-Hot** sobre `color` y `ciudad`.
     - `handle_unknown="ignore"` evita errores si aparece una **categoría nueva** en datos futuros (esa fila queda con ceros en esas dummies).
     - `sparse_output=False` devuelve salida **densa** (cómodo para DataFrame).
   - `("ordinal", OrdinalEncoder(...), cols_ordinal)`: codifica ordinalmente `nivel_educativo` y `satisfaccion`.
     - `categories=categorias_ordinales` fija el **orden** (p. ej., *Primaria < Secundaria < …*).
     - `handle_unknown="use_encoded_value", unknown_value=-1` mapea **toda categoría no vista** a **-1** (fácil de identificar).
   - `remainder="passthrough"`: **todas las columnas no listadas** (ej. `dias_desde_registro`) **pasan intactas** a la salida.
   - `verbose_feature_names_out=False`: conserva **nombres “limpios”** en las columnas resultantes.

4) **Ajuste y transformación**
   - `preprocesador.fit(datos)`: el objeto **aprende** lo necesario (medias, min/máx, medianas/IQR, vocabularios y órdenes de categorías).
   - `preprocesador.transform(datos)`: **aplica** esas transformaciones a `datos` y devuelve una matriz con las columnas ya procesadas y las que pasan por `passthrough`.

5) **Reconstrucción de `DataFrame` final**
   - `get_feature_names_out()` recupera los **nombres de las columnas** en el orden de salida (por bloques: primero `std`, luego `minmax`, luego `robust`, `onehot`, `ordinal`, y al final las de `passthrough`).
   - Se crea `df_proc` con dichas columnas y el mismo índice que `datos`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer

# Cargar datos
datos = pd.read_csv("datos_escalado_y_codificacion.csv")

In [ ]:
# Nombres de las columnas
datos.columns

In [ ]:
# Visualizar los primeros 5 renglones
datos.head()

In [ ]:
# Capturar columnas con atípicos

num_cols = datos.select_dtypes(include=[np.number]).columns

cols_con_outliers = []
for c in num_cols:
    s = pd.to_numeric(datos[c], errors="coerce").dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = int(((s < lim_inf) | (s > lim_sup)).sum())
    if n_out > 0:
        cols_con_outliers.append(c)

cols_con_outliers

In [ ]:
# Definir columnas
cols_std    = ["peso_kg"]                   # StandardScaler
cols_minmax = ["humedad_relativa_pct", "calificacion_examen"]   # MinMaxScaler
cols_robust = ["temperatura_corp", "ingreso_mensual", "gastos_publicidad"]          # RobustScaler
cols_onehot  = ["color", "ciudad"]                              # One-Hot
cols_ordinal = ["nivel_educativo", "satisfaccion"]              # Ordinal

# Orden explícito para las ordinales
categorias_ordinales = [
    ["Primaria", "Secundaria", "Preparatoria", "Universidad", "Posgrado"],  # nivel_educativo
    ["Baja", "Media", "Alta"],                                              # satisfaccion
]

In [ ]:
# Construir ColumnTransformer
preprocesador = ColumnTransformer(
    transformers=[
        ("std",    StandardScaler(), cols_std),
        ("minmax", MinMaxScaler(),  cols_minmax),
        ("robust", RobustScaler(),  cols_robust),
        ("onehot", OneHotEncoder(sparse_output=False, drop=None, handle_unknown="ignore"), cols_onehot),
        ("ordinal", OrdinalEncoder(categories=categorias_ordinales,
                                   handle_unknown="use_encoded_value", unknown_value=-1), cols_ordinal),
    ],
    remainder="passthrough",          # Así vemos qué pasa con columnas NO procesadas
    verbose_feature_names_out=False
)

# Ajustar (fit) y transformar
preprocesador.fit(datos)
X_proc = preprocesador.transform(datos)

# Convertir a DataFrame con nombres de columnas
cols_out = preprocesador.get_feature_names_out()
df_proc = pd.DataFrame(X_proc, columns=cols_out, index=datos.index)

print(df_proc.head())


In [ ]:
# Guardar la tabla ya procesada

df_proc.to_csv("datos_escalados_y_codificados.csv",index=False)
